# Task 3: Correlation Between News Sentiment and Stock Movement

This notebook aligns news timestamps with trading days, scores headline sentiment, aggregates multiple same-day articles, computes daily percentage returns, calculates Pearson correlation, and visualizes the relationship.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.correlation import aggregate_daily_sentiment, map_to_next_trading_day, pearson_correlation, prepare_correlation_frame
from src.data_loading import clean_stock_prices, load_news_data, load_stock_prices, REQUIRED_PRICE_COLUMNS
from src.indicators import daily_returns
from src.sentiment import sentiment_category

sns.set_theme(style='whitegrid')
TICKER = 'AAPL'
START_DATE = '2018-01-01'
END_DATE = '2020-06-30'

## Sentiment Tool Selection

TextBlob is selected because it is lightweight, fast for headline-level polarity scoring, and returns a continuous polarity score from -1 to 1. VADER would also be appropriate for short social/news text; TextBlob is used here for simple reproducibility. If TextBlob is unavailable, the repository fallback lexicon scorer keeps the pipeline executable.

In [ ]:
news = load_news_data()
news = news[news['stock'].eq(TICKER)].copy()
news['headline'] = news['headline'].fillna('').astype(str)

try:
    from textblob import TextBlob
    news['sentiment_score'] = news['headline'].map(lambda text: TextBlob(text).sentiment.polarity)
    sentiment_note = 'TextBlob polarity scores used.'
except Exception as exc:
    from src.sentiment import lexicon_sentiment
    news['sentiment_score'] = news['headline'].map(lexicon_sentiment)
    sentiment_note = f'Fallback lexicon sentiment used because TextBlob was unavailable: {exc}'

news['sentiment_category'] = news['sentiment_score'].map(sentiment_category)
print(sentiment_note)
display(news[['date', 'stock', 'headline', 'sentiment_score', 'sentiment_category']].head())

## Load Prices and Compute Daily Returns

Daily percentage return is computed from adjusted close prices using `(Close_t - Close_{t-1}) / Close_{t-1} * 100`. The adjusted close series is used so splits and dividends do not distort the return calculation.

In [ ]:
try:
    prices = load_stock_prices(TICKER)
    price_source = f'Loaded local data/stock_prices/{TICKER}.csv'
except FileNotFoundError:
    import yfinance as yf
    min_news_date = news['date'].min().date().isoformat()
    max_news_date = (news['date'].max() + pd.Timedelta(days=7)).date().isoformat()
    prices = yf.download(TICKER, start=min_news_date, end=max_news_date, auto_adjust=False, progress=False)
    price_source = f'Downloaded {TICKER} prices with yfinance'

prices = clean_stock_prices(prices[REQUIRED_PRICE_COLUMNS])
returns = pd.DataFrame({
    'trading_day': prices.index,
    'daily_return_pct': daily_returns(prices['Adj Close']).values,
}).dropna()

print(price_source)
display(returns.head())

## Date Alignment

News timestamps are normalized to calendar dates in UTC. Articles published on weekends or market holidays are mapped to the next available trading day in the price index, so they can be compared with the next market return.

In [ ]:
news['trading_day'] = map_to_next_trading_day(news['date'], prices.index)
daily_sentiment = aggregate_daily_sentiment(news)
corr_frame = prepare_correlation_frame(daily_sentiment, returns, TICKER)
corr_value = pearson_correlation(corr_frame)

print(f'Rows after daily alignment: {len(corr_frame):,}')
print(f'Pearson correlation: {corr_value:.4f}')
display(corr_frame.head())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.regplot(data=corr_frame, x='avg_sentiment', y='daily_return_pct', ax=ax, scatter_kws={'alpha': 0.65})
ax.set_title(f'{TICKER}: Average Daily Sentiment vs Daily Return')
ax.set_xlabel('Average Daily Sentiment Score')
ax.set_ylabel('Daily Return (%)')
ax.annotate(f'Pearson r = {corr_value:.3f}', xy=(0.05, 0.95), xycoords='axes fraction', va='top', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
category_returns = (
    corr_frame.groupby('dominant_category', as_index=False)['daily_return_pct']
    .mean()
    .rename(columns={'daily_return_pct': 'avg_daily_return_pct'})
)

ax = sns.barplot(data=category_returns, x='dominant_category', y='avg_daily_return_pct', palette='Set2')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'{TICKER}: Average Daily Return by Sentiment Category')
ax.set_xlabel('Sentiment Category')
ax.set_ylabel('Average Daily Return (%)')
plt.tight_layout()
plt.show()

display(category_returns)

## Results Interpretation

The Pearson coefficient above measures the linear association between same-day average headline sentiment and adjusted-close daily return. A positive value means more positive news days tended to align with higher returns, while a negative value means more positive news days tended to align with lower returns. In most headline datasets this relationship is expected to be weak because prices incorporate many other signals, headlines can be reactive rather than predictive, and single-day returns are noisy.

Important limitations remain. Some articles are published before market open, during market hours, or after close, so a same/next-trading-day mapping can blur the true reaction window. Lag effects may exist beyond one trading day, and confounding factors such as earnings announcements, macro shocks, sector moves, analyst target changes, and overall market beta can dominate the sentiment signal. A stronger study would test multiple lags, control for market returns, and separate pre-market from post-market publication timestamps.